<a href="https://colab.research.google.com/github/s251198/CS521_das44_HW3/blob/main/P2_CS521.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install syncode transformers torch
%pip install accelerate


In [ ]:
import re
import torch
from syncode import Syncode

extraction_grammar = r"""
start: "Operands: " number " and " number "\nOperator: " operator

number: /[0-9]+(\.[0-9]+)?/
operator: "+" | "-" | "*" | "/"
"""


In [ ]:
syn_model = Syncode(model = "microsoft/phi-2", grammar = extraction_grammar, device = "cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
FEW_SHORT_PROMPT = """Extract exactly two operands and the binary operator from the query.
Copy each operand exactly as written, preserving every digit after the decimal point, including trailing zeros. Do not round, shorten, or calculate the answer. Return only the two-line format shown in the examples.

Query: What is 327. multiplied by 11.0?
Operands: 327.0 and 11.0
Operator: *

Query: What is 45.1 plus 23.54?
Operands: 45.1 and 23.54
Operator: +

Query: What is 120.4 divided by 4.0?
Operands: 120.4 and 4.0
Operator: /

Query: What is 18.06 multiplied by 3.20?
Operands: 18.06 and 3.20
Operator: *
"""

In [ ]:
def run_syncode(textual_input: str):
    """
    Takes a textual math query, runs SynCode for structured extraction,
    and evaluates the extracted expression using Python.

    Returns:
        tuple: (ordered_list_of_operands, operator_str, final_answer_float)
    """
    query_operands = re.findall(r"\d+(?:\.\d*)?", textual_input)
    if len(query_operands) != 2:
        raise ValueError(f"Expected exactly two numeric operands in query: {textual_input!r}")
    expected_operands = [float(value) for value in query_operands]
    response = ""
    operands = None
    op_str = None

    for attempt in range(2):
        correction = ""
        if attempt:
            correction = (
                "Correction: The previous extraction did not match the query. "
                "Read the query again and copy both operands exactly, including all decimal digits. "
                "Do not round, shorten, or calculate. Return only the two required lines.\n"
            )
        prompt = f"{FEW_SHORT_PROMPT}\n{correction}Query: {textual_input}\n"
        completions = syn_model.infer(prompt)
        if not completions:
            raise RuntimeError("Syncode returned no completion for the prompt")
        response = completions[0].strip()

        operands_match = re.search(
            r"Operands:\s*([0-9]+(?:\.[0-9]+)?)\s*and\s*([0-9]+(?:\.[0-9]+)?)",
            response,
        )
        operator_match = re.search(r"Operator:\s*([+\-*/])", response)
        if operands_match and operator_match:
            extracted_operands = [float(operands_match.group(1)), float(operands_match.group(2))]
            if extracted_operands == expected_operands:
                operands = extracted_operands
                op_str = operator_match.group(1)
                break

    if operands is None or op_str is None:
        raise ValueError(
            f"Syncode could not extract the query correctly after two attempts. "
            f"Expected operands {expected_operands}; last completion was {response!r}"
        )
    op1, op2 = operands

    # Evaluate the binary operation in Python interpreter
    if op_str == "+":
        final_answer = op1 + op2
    elif op_str == "-":
        final_answer = op1 - op2
    elif op_str == "*":
        final_answer = op1 * op2
    elif op_str == "/":
        final_answer = op1 / op2 if op2 != 0 else float("inf")
    else:
        raise ValueError(f"Unsupported operator: {op_str}")

    return operands, op_str, float(final_answer)

In [ ]:
if __name__ == "__main__":
    test_queries = [
        "What is 200 multiplied by 23.20?",
        "What is 20 plus 10?",
        "What is 4222 divided by 4.0?"
    ]

    for query in test_queries:
        operands, operator, answer = run_syncode(query)
        print(f"Query: {query}")
        print(f"  -> Extracted Operands: {operands}")
        print(f"  -> Extracted Operator: {operator}")
        print(f"  -> Final Answer: {answer}\n")